# ML Testing & MLOps: Building a Pre-Deployment Model Gate

**Dataset:** [Palmer Station Penguins](https://github.com/allisonhorst/palmerpenguins) — 344 real
morphological measurements of three penguin species (Adelie, Chinstrap, Gentoo) collected at
Palmer Station, Antarctica (bill length/depth, flipper length, body mass, island), loaded from the
`mwaskom/seaborn-data` CSV mirror.

## Problem statement

You are the QA engineer responsible for gating every retrain of a penguin-species classifier before
it is allowed to reach production. The model predicts species from four physical measurements taken
by field biologists. A sloppy field-measurement pipeline means you can't just check "is accuracy
above X" and call it a day — the CI gate also has to catch: malformed input batches (values outside
a physically plausible range, or missing data above a tolerance), a model that silently collapses on
the rarer Chinstrap species or the rarer Torgersen island even while overall accuracy looks fine,
training that isn't actually reproducible run-to-run, and a sensor recalibration that shifts the
incoming feature distribution relative to what the model was trained on.

## What you should produce

By the end of this notebook you should have:

- `EXPECTED_SCHEMA`, a dict describing each feature's expected dtype and a physically plausible
  value range, and `validate_schema(df, schema, max_null_rate=0.05)` returning a list of
  human-readable violation strings (empty list == passed).
- `evaluate_slices(model, X, y, slice_defs)` returning a `DataFrame` of `n`, `accuracy`, and
  `macro_f1` for `"overall"` plus every named slice (by species, by island).
- `check_reproducibility(build_fn, X_test, y_test, seeds)` that retrains across at least 3 seeds via
  `build_fn(seed)` and returns `(scores, std_of_scores)` for macro-F1 on the held-out test set.
- `detect_drift(reference, current, feature_cols, alpha=0.01)` using a two-sample
  Kolmogorov–Smirnov test per feature (`scipy.stats.ks_2samp`), returning per-feature statistic,
  p-value, and a `drifted` boolean.
- `run_gate(...)` that runs all four checks against fixed thresholds and returns
  `{"passed": bool, "failures": [str, ...]}`, where every failure string names which check failed
  and why.
- A demonstration: build a `drifted_batch` by shifting `bill_length_mm` (+4mm) and
  `flipper_length_mm` (×1.05) on a copy of the test features — simulating a caliper
  miscalibration — then show `run_gate(...)` **passes** on the clean test batch and **fails**
  (naming the specific check(s) that tripped) on the drifted batch.

---

## Starter cells

Fill in every `# TODO`. Don't look at the solution until you've tried each step.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import ks_2samp

RANDOM_STATE = 42
pd.set_option("display.precision", 3)

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(url)

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# A handful of rows are missing every measurement (not just `sex`) -- drop those for this
# exercise, same as the original palmerpenguins "complete cases" convention.
penguins = penguins.dropna(subset=["species", "island"] + FEATURES).reset_index(drop=True)

print(penguins.shape)
penguins[["species", "island"] + FEATURES].describe(include="all")

### Part 1 — schema validation

Before any model touches a batch, a production pipeline should sanity-check it against a
contract: right columns, numeric, no excessive nulls, values within a physically plausible range.
This is the cheapest check to run and the first thing that catches bad upstream data silently
poisoning every number downstream of it.

In [ ]:
# TODO: build EXPECTED_SCHEMA, a dict keyed by column name, where each value is a dict with:
#   - "min": a plausible real-world lower bound for that measurement
#   - "max": a plausible real-world upper bound for that measurement
# (Look at penguins[FEATURES].describe() above and give yourself a little headroom beyond the
#  observed min/max -- you're encoding "physically plausible", not "exactly what we've seen".)
# Columns: bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
EXPECTED_SCHEMA = {
    # "bill_length_mm": {"min": ..., "max": ...},
}


# Return a list of human-readable violation strings; an empty list means the batch passed.
# For each column in `schema`, check (in order):
#   1. the column exists in `df`
#   2. `df[col]` is numeric (pd.api.types.is_numeric_dtype)
#   3. the null rate for that column is <= max_null_rate
#   4. no non-null value falls outside [schema[col]["min"], schema[col]["max"]]
def validate_schema(df, schema, max_null_rate=0.05):
    violations = []
    # TODO
    return violations


validate_schema(penguins[FEATURES], EXPECTED_SCHEMA)

### Part 2 — slice-based evaluation

Chinstrap is the minority species (68/342 rows) and Torgersen is the minority island (52/342 rows).
A model can post a great overall accuracy while quietly failing on exactly the slice a biologist
cares about most. Measure every slice, not just the headline number.

In [ ]:
# TODO: write evaluate_slices(model, X, y, slice_defs), where slice_defs is a dict mapping a
# slice name -> a boolean mask (pd.Series, same index as X/y) selecting the rows in that slice.
#
# For "overall" (all of X/y) and for every entry in slice_defs, compute:
#   - n: the slice size
#   - accuracy: accuracy_score on that slice
#   - macro_f1: f1_score(..., average="macro") on that slice
# Return a DataFrame indexed by slice name (with "overall" as the first row) with those 3 columns.
# Skip any slice with zero rows.
def evaluate_slices(model, X, y, slice_defs):
    rows = {}
    # TODO
    return pd.DataFrame(rows).T


# TODO: split penguins into X_train, X_test, y_train, y_test using FEATURES as X and "species" as
# y. Use test_size=0.25, stratify=y, random_state=RANDOM_STATE.

# TODO: train a RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE) on
# (X_train, y_train) and call it `model`.

# TODO: build slice_defs with one entry per species value in y_test ("species=<name>") and one
# entry per island value among the test rows ("island=<name>"), then call
# evaluate_slices(model, X_test, y_test, slice_defs) and look at the result.

### Part 3 — reproducibility check

A pass/fail gate is only as trustworthy as the number it gates on. If swapping the random seed
moves macro-F1 by more than ordinary noise, a single run's score can't be trusted as a gate
input — this is the ML-training equivalent of a flaky test.

In [ ]:
# TODO: write check_reproducibility(build_fn, X_test, y_test, seeds).
# `build_fn(seed)` trains and returns a fitted model for that random_state (same train split each
# time -- only the model's own seed changes).
# For each seed in `seeds`: build the model, predict on X_test, score macro-F1.
# Return (scores, std_of_scores) where scores is the list of per-seed macro-F1 and std_of_scores
# is float(np.std(scores)).
def check_reproducibility(build_fn, X_test, y_test, seeds):
    scores = []
    # TODO
    return scores, float(np.std(scores))


# TODO: define build_model(seed) that trains RandomForestClassifier(n_estimators=200,
# random_state=seed) on (X_train, y_train) and returns it, then call
# check_reproducibility(build_model, X_test, y_test, seeds=[1, 2, 3, 4, 5]).

### Part 4 — drift detection

Even a perfectly reproducible, perfectly validated-at-training-time model can fail in production
if the incoming feature distribution has shifted since training (sensor recalibration, a new data
source, a seasonal effect). A two-sample Kolmogorov–Smirnov test per feature is a simple,
model-agnostic way to flag that.

In [ ]:
# TODO: write detect_drift(reference, current, feature_cols, alpha=0.01).
# For each column in feature_cols, run scipy.stats.ks_2samp(reference[col].dropna(),
# current[col].dropna()) to get (statistic, p_value).
# A feature counts as drifted if p_value < alpha.
# Return a dict: {col: {"statistic": float, "p_value": float, "drifted": bool}}
def detect_drift(reference, current, feature_cols, alpha=0.01):
    report = {}
    # TODO
    return report


# TODO: sanity-check it on data that should NOT look drifted: detect_drift(X_train, X_test,
# FEATURES). Every feature should come back drifted=False.

### Part 5 — the gate

Wire the four checks into one aggregator a CI job could call with a single line.

In [ ]:
# TODO: write run_gate(model, build_fn, X_test, y_test, schema, seeds,
#                      reference_df, current_df, feature_cols,
#                      min_accuracy=0.90, min_slice_f1=0.75, max_seed_std=0.03, drift_alpha=0.01)
# that runs, in order:
#   1. validate_schema(current_df, schema)                      -> prefix failures with "schema: "
#   2. overall accuracy of model on (X_test, y_test) vs min_accuracy
#        -> failure string like f"overall accuracy={acc:.3f} below min_accuracy={min_accuracy}"
#   3. evaluate_slices by species on (X_test, y_test); any slice (excluding "overall") with
#      macro_f1 < min_slice_f1 is a failure, e.g.
#        f"slice {name} macro_f1={f1:.3f} below min_slice_f1={min_slice_f1}"
#   4. check_reproducibility(build_fn, X_test, y_test, seeds); std > max_seed_std is a failure
#   5. detect_drift(reference_df, current_df, feature_cols, drift_alpha); any drifted feature is
#      a failure, e.g. f"drift detected in {col} (p={p_value:.4g})"
# Return {"passed": len(failures) == 0, "failures": failures}.
def run_gate(model, build_fn, X_test, y_test, schema, seeds,
             reference_df, current_df, feature_cols,
             min_accuracy=0.90, min_slice_f1=0.75, max_seed_std=0.03, drift_alpha=0.01):
    failures = []
    # TODO
    return {"passed": len(failures) == 0, "failures": failures}

### Part 6 — simulate a drifted production batch

Simulate a caliper miscalibration hitting the production data pipeline, and show the gate tells
the difference between a clean batch and a drifted one.

In [ ]:
# TODO:
# 1. Build `drifted_batch`: a copy of X_test where bill_length_mm has +4 added to every value and
#    flipper_length_mm is multiplied by 1.05 (leave bill_depth_mm and body_mass_g unchanged).
# 2. Run run_gate(...) with current_df=X_test (clean) and reference_df=X_train, and separately with
#    current_df=drifted_batch. Use seeds=[1, 2, 3].
# 3. Print both results. You should see passed=True for the clean batch and passed=False for the
#    drifted batch, with the failures list naming the drift (and possibly schema) checks that
#    caught it -- not a vague "something's wrong".

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import ks_2samp

RANDOM_STATE = 42

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(url)
FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins = penguins.dropna(subset=["species", "island"] + FEATURES).reset_index(drop=True)

EXPECTED_SCHEMA = {
    "bill_length_mm": {"min": 25.0, "max": 65.0},
    "bill_depth_mm": {"min": 10.0, "max": 25.0},
    "flipper_length_mm": {"min": 165.0, "max": 240.0},
    "body_mass_g": {"min": 2500.0, "max": 6500.0},
}

def validate_schema(df, schema, max_null_rate=0.05):
    violations = []
    for col, spec in schema.items():
        if col not in df.columns:
            violations.append(f"missing column: {col}")
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            violations.append(f"{col}: expected numeric dtype, got {df[col].dtype}")
            continue
        null_rate = df[col].isna().mean()
        if null_rate > max_null_rate:
            violations.append(f"{col}: null rate {null_rate:.2%} exceeds {max_null_rate:.2%}")
        values = df[col].dropna()
        out_of_range = ((values < spec["min"]) | (values > spec["max"])).sum()
        if out_of_range:
            violations.append(
                f"{col}: {out_of_range} values outside [{spec['min']}, {spec['max']}]"
            )
    return violations

X = penguins[FEATURES]
y = penguins["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

def build_model(seed):
    model = RandomForestClassifier(n_estimators=200, random_state=seed)
    model.fit(X_train, y_train)
    return model

model = build_model(RANDOM_STATE)
print("held-out accuracy:", accuracy_score(y_test, model.predict(X_test)))

def evaluate_slices(model, X, y, slice_defs):
    rows = {}
    preds_all = model.predict(X)
    rows["overall"] = {
        "n": len(y),
        "accuracy": accuracy_score(y, preds_all),
        "macro_f1": f1_score(y, preds_all, average="macro"),
    }
    for name, mask in slice_defs.items():
        if mask.sum() == 0:
            continue
        preds = model.predict(X[mask])
        rows[name] = {
            "n": int(mask.sum()),
            "accuracy": accuracy_score(y[mask], preds),
            "macro_f1": f1_score(y[mask], preds, average="macro"),
        }
    return pd.DataFrame(rows).T

island_test = penguins.loc[X_test.index, "island"]
slice_defs = {f"species={sp}": (y_test == sp) for sp in y_test.unique()}
slice_defs.update({f"island={isl}": (island_test == isl) for isl in island_test.unique()})
print(evaluate_slices(model, X_test, y_test, slice_defs))

def check_reproducibility(build_fn, X_test, y_test, seeds):
    scores = [f1_score(y_test, build_fn(seed).predict(X_test), average="macro") for seed in seeds]
    return scores, float(np.std(scores))

scores, std = check_reproducibility(build_model, X_test, y_test, [1, 2, 3, 4, 5])
print("per-seed macro-F1:", scores, "std:", std)

def detect_drift(reference, current, feature_cols, alpha=0.01):
    report = {}
    for col in feature_cols:
        stat, p = ks_2samp(reference[col].dropna(), current[col].dropna())
        report[col] = {"statistic": float(stat), "p_value": float(p), "drifted": bool(p < alpha)}
    return report

def run_gate(model, build_fn, X_test, y_test, schema, seeds,
             reference_df, current_df, feature_cols,
             min_accuracy=0.90, min_slice_f1=0.75, max_seed_std=0.03, drift_alpha=0.01):
    failures = []

    for v in validate_schema(current_df, schema):
        failures.append(f"schema: {v}")

    acc = accuracy_score(y_test, model.predict(X_test))
    if acc < min_accuracy:
        failures.append(f"overall accuracy={acc:.3f} below min_accuracy={min_accuracy}")

    species_slices = {f"species={sp}": (y_test == sp) for sp in y_test.unique()}
    for name, row in evaluate_slices(model, X_test, y_test, species_slices).iterrows():
        if name != "overall" and row["macro_f1"] < min_slice_f1:
            failures.append(
                f"slice {name} macro_f1={row['macro_f1']:.3f} below min_slice_f1={min_slice_f1}"
            )

    _, std = check_reproducibility(build_fn, X_test, y_test, seeds)
    if std > max_seed_std:
        failures.append(f"reproducibility: macro_f1 std across seeds={std:.4f} exceeds {max_seed_std}")

    for col, info in detect_drift(reference_df, current_df, feature_cols, drift_alpha).items():
        if info["drifted"]:
            failures.append(f"drift detected in {col} (p={info['p_value']:.4g})")

    return {"passed": len(failures) == 0, "failures": failures}

drifted_batch = X_test.copy()
drifted_batch["bill_length_mm"] = drifted_batch["bill_length_mm"] + 4.0
drifted_batch["flipper_length_mm"] = drifted_batch["flipper_length_mm"] * 1.05

clean_result = run_gate(
    model, build_model, X_test, y_test, EXPECTED_SCHEMA, seeds=[1, 2, 3],
    reference_df=X_train, current_df=X_test, feature_cols=FEATURES,
)
drifted_result = run_gate(
    model, build_model, X_test, y_test, EXPECTED_SCHEMA, seeds=[1, 2, 3],
    reference_df=X_train, current_df=drifted_batch, feature_cols=FEATURES,
)
print("CLEAN BATCH:", clean_result)
print("DRIFTED BATCH:", drifted_result)
```

**Why this matters for ML quality engineering:** none of these four checks subsumes another.
Schema validation catches malformed *inputs* before they ever reach a metric; slice evaluation
catches a model that is "fine on average" but broken for the group you actually care about;
the reproducibility check catches the case where the number you're gating on is itself noise,
not signal; and drift detection catches a production population that no longer looks like the
one the model was validated against, *before* that shows up as a performance regression you'd
otherwise only see after the damage is done. A real CI gate for a model is this combination, not
any single number -- which is exactly the generalization of "build gate" thinking an SQA
background already has, applied to a system whose behavior depends on data as well as code.
</details>